# Step 1: Integration and main cell-type labeling

Processes and integrates multiple Xenium samples and assigns coarse cell-type
labels. Integration largely follows Yu et al. 2025
(https://doi.org/10.1038/s41588-025-02158-6).

Reads raw Xenium output directories and writes the integrated AnnData object
(`xenium_integrated.h5ad`) together with per-celltype subclustered AnnData
files used by Step 2. A per-cluster marker-gene table is also exported to
support manual cell-type annotation.

## Setup and imports

In [ ]:
import scanpy as sc
import squidpy as sq
from spatialdata_io import xenium
import spatialdata as sd
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from pathlib import Path
import anndata as ad
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import glob
import os
import shutil

In [ ]:
sc.settings.verbosity = 3
np.random.seed(26)

In [ ]:
# Path config
indir = '/path/to/xenium/raw_data/'                # Available upon request
outdir_h5ad = '/path/to/integrated/data/' # Available upon request

## Helper functions

In [ ]:
def fix_morphology_focus_naming(xenium_path):
    """Fix morphology focus naming by temporarily hiding original files.
    Required for spatialdata_io.xenium() compatibility."""
    morphology_dir = os.path.join(xenium_path, "morphology_focus")
    if not os.path.exists(morphology_dir):
        print("Morphology focus directory not found - skipping fix")
        return False

    backup_dir = os.path.join(morphology_dir, "backup_original")
    os.makedirs(backup_dir, exist_ok=True)

    original_files = [
        'ch0000_dapi.ome.tif',
        'ch0001_atp1a1_cd45_e-cadherin.ome.tif',
        'ch0002_18s.ome.tif',
        'ch0003_alphasma_vimentin.ome.tif'
    ]
    for orig_file in original_files:
        orig_path = os.path.join(morphology_dir, orig_file)
        backup_path = os.path.join(backup_dir, orig_file)
        if os.path.exists(orig_path):
            shutil.move(orig_path, backup_path)
            print(f"Moved to backup: {orig_file}")

    links = {
        'morphology_focus_0000.ome.tif': 'backup_original/ch0000_dapi.ome.tif',
        'morphology_focus_0001.ome.tif': 'backup_original/ch0001_atp1a1_cd45_e-cadherin.ome.tif',
        'morphology_focus_0002.ome.tif': 'backup_original/ch0002_18s.ome.tif',
        'morphology_focus_0003.ome.tif': 'backup_original/ch0003_alphasma_vimentin.ome.tif'
    }
    for link_name, target_path in links.items():
        link_path = os.path.join(morphology_dir, link_name)
        full_target_path = os.path.join(morphology_dir, target_path)
        if os.path.exists(full_target_path) and not os.path.exists(link_path):
            os.symlink(target_path, link_path)
            print(f"Created link: {link_name} -> {target_path}")
    return True

def restore_morphology_focus_naming(xenium_path):
    morphology_dir = os.path.join(xenium_path, "morphology_focus")
    backup_dir = os.path.join(morphology_dir, "backup_original")
    if os.path.exists(backup_dir):
        for i in range(4):
            link_path = os.path.join(morphology_dir, f"morphology_focus_000{i}.ome.tif")
            if os.path.islink(link_path):
                os.unlink(link_path)
        for orig_file in os.listdir(backup_dir):
            shutil.move(os.path.join(backup_dir, orig_file), os.path.join(morphology_dir, orig_file))
        shutil.rmtree(backup_dir)

## Load individual samples
Loop over all per-sample directories under `indir` and save each as a `.h5ad`.

In [ ]:
sample_dirs = sorted([p for p in glob.glob(os.path.join(indir, "*")) if os.path.isdir(p)])
if len(sample_dirs) == 0:
    raise ValueError(f"No sample directories found in {indir}")

for xenium_path in sample_dirs:
    sampleName = os.path.basename(xenium_path.rstrip("/"))
    print(f"Processing {sampleName}...")
    fix_morphology_focus_naming(xenium_path)
    sdata = xenium(xenium_path)
    adata = sdata.tables["table"]
    out_file = os.path.join(outdir_h5ad, f"anndata_{sampleName}.h5ad")
    adata.write_h5ad(out_file, compression="gzip")
    print(f"Saved: {out_file}")

## Combine samples

In [ ]:
annFileArray = sorted(glob.glob(outdir_h5ad + "anndata_*.h5ad"))
ann_list = []
for filename in annFileArray:
    print(f"Loading: {filename}")
    ann = sc.read_h5ad(filename)
    sampleName = Path(filename).stem.replace('anndata_', '')
    ann.obs['sampleName'] = sampleName
    ann_list.append(ann)

In [ ]:
sample_keys = [Path(f).stem.replace('anndata_', '') for f in annFileArray]
ann_comb = ad.concat(
    ann_list,
    join='outer',
    axis=0,
    label="sample",
    keys=sample_keys,
    merge='same'
)
print(f"Combined dataset: {ann_comb.n_obs} cells x {ann_comb.n_vars} features")

## Feature selection

In [ ]:
features = ann_comb.var.index
sele_features = features[~features.str.contains(r'>|_WT')].tolist()
ann_comb.var['is_gene'] = ~ann_comb.var_names.str.contains(r'>|_WT|clone')
print(f"Selected features: {len(sele_features)} / {len(features)}")

## Quality control

In [ ]:
sc.pp.calculate_qc_metrics(ann_comb, percent_top=(10, 20, 50, 150), inplace=True)
sc.pp.calculate_qc_metrics(ann_comb, percent_top=(10, 20, 50, 150), inplace=True, qc_vars=['is_gene'])

cprobes = (ann_comb.obs["control_probe_counts"].sum() / ann_comb.obs["total_counts"].sum() * 100)
cwords = (ann_comb.obs["control_codeword_counts"].sum() / ann_comb.obs["total_counts"].sum() * 100)
print(f"Negative DNA probe count %: {cprobes:.2f}")
print(f"Negative decoding count %: {cwords:.2f}")

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))
axs[0].set_title("Total transcripts per cell")
sns.histplot(ann_comb.obs["total_counts"], kde=False, ax=axs[0])
axs[0].set_xlim(0, 200)
axs[1].set_title("Unique transcripts per cell")
sns.histplot(ann_comb.obs["n_genes_by_counts"], kde=False, ax=axs[1])
axs[2].set_title("Area of segmented cells")
sns.histplot(ann_comb.obs["cell_area"], kde=False, ax=axs[2])
axs[3].set_title("Nucleus ratio")
sns.histplot(ann_comb.obs["nucleus_area"] / ann_comb.obs["cell_area"], kde=False, ax=axs[3])
plt.tight_layout()
plt.show()

## Filtering and normalization

In [ ]:
ann_comb.layers["raw_counts"] = ann_comb.X.copy()
print(f"Before filtering: {ann_comb.n_obs} cells, {ann_comb.n_vars} genes")
sc.pp.filter_cells(ann_comb, min_counts=20)
sc.pp.filter_genes(ann_comb, min_cells=30)
print(f"After filtering: {ann_comb.n_obs} cells, {ann_comb.n_vars} genes")

sc.pp.normalize_total(ann_comb, target_sum=1e4, inplace=True)
sc.pp.log1p(ann_comb)

ann_comb.var["highly_variable"] = ann_comb.var.index.isin(sele_features)
print(f"Highly variable genes: {ann_comb.var['highly_variable'].sum()}")

## Spatial QC and coordinate verification

In [ ]:
assert 'spatial' in ann_comb.obsm, "Missing 'spatial' coordinates in obsm"
coords = ann_comb.obsm['spatial']
print("=== Spatial Coordinate Verification ===")
print(f"Total cells: {ann_comb.n_obs}")
print(f"Coordinate shape: {coords.shape}")
print(f"Coordinate range X: [{coords[:, 0].min():.1f}, {coords[:, 0].max():.1f}]")
print(f"Coordinate range Y: [{coords[:, 1].min():.1f}, {coords[:, 1].max():.1f}]")

duplicates = pd.DataFrame(coords).duplicated().sum()
nan_coords = np.isnan(coords).any(axis=1).sum()
inf_coords = np.isinf(coords).any(axis=1).sum()
print(f"Duplicate coordinates: {duplicates} ({duplicates/ann_comb.n_obs*100:.2f}%)")
print(f"NaN coordinates: {nan_coords}")
print(f"Inf coordinates: {inf_coords}")

## Integration and batch correction

In [ ]:
RANDOM_STATE = 26
N_PCS = 20
N_PCS_SUBTYPE = 15
N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.2

In [ ]:
sc.pp.pca(ann_comb, n_comps=N_PCS, random_state=RANDOM_STATE)
sc.pp.neighbors(ann_comb, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS, random_state=RANDOM_STATE)
sc.tl.umap(ann_comb, min_dist=UMAP_MIN_DIST, random_state=RANDOM_STATE)

sc.external.pp.harmony_integrate(ann_comb, 'sample')

sc.pp.neighbors(ann_comb, use_rep="X_pca_harmony", n_neighbors=N_NEIGHBORS, random_state=RANDOM_STATE)
sc.tl.umap(ann_comb, min_dist=UMAP_MIN_DIST, random_state=RANDOM_STATE)
sc.tl.leiden(ann_comb, resolution=0.8, key_added='leiden_0.8')

In [ ]:
# Differential expression for cell-type annotation
sc.tl.rank_genes_groups(ann_comb, 'leiden_0.8', method='wilcoxon', key_added="wilcoxon")
sc.pl.rank_genes_groups(ann_comb, n_genes=10, sharey=False, key="wilcoxon")

# Export marker gene table (used to build the celltype map below)
df_genes = pd.DataFrame(ann_comb.uns['wilcoxon']['names'])

# Remove contamination cluster (chosen from marker inspection)
ann_comb = ann_comb[ann_comb.obs['leiden_0.8'] != '4'].copy()

ann_comb.obs["celltype"] = ann_comb.obs["leiden_0.8"].map({
    "0": "Neuroblast", "1": "Neuroblast", "2": "Neuroblast", "3": "Neuroblast",
    "5": "T", "6": "Fibroblast", "7": "Neuroblast", "8": "Macrophage",
    "9": "Endothelial", "10": "Macrophage", "11": "Schwann", "12": "B",
    "13": "Neuroblast", "14": "Neuroblast"
})

## Subclustering of major cell types

In [ ]:
celltypes = ["Neuroblast", "Macrophage", "B", "T"]
resolutions = {"Neuroblast": 0.5, "Macrophage": 0.3, "B": 0.3, "T": 0.5}

for ct in celltypes:
    print(f"\nSubclustering {ct}...")
    ad_sub = ann_comb[ann_comb.obs["celltype"] == ct].copy()
    if ad_sub.n_obs < 100:
        print(f"  Skipping {ct}: insufficient cells ({ad_sub.n_obs})")
        continue
    sc.pp.pca(ad_sub, n_comps=N_PCS_SUBTYPE, random_state=RANDOM_STATE)
    sc.pp.neighbors(ad_sub, n_neighbors=N_NEIGHBORS, n_pcs=N_PCS_SUBTYPE, random_state=RANDOM_STATE)
    sc.tl.umap(ad_sub, min_dist=UMAP_MIN_DIST, random_state=RANDOM_STATE)
    res = resolutions[ct]
    key = f"leiden_{ct}_{res}"
    sc.tl.leiden(ad_sub, resolution=res, key_added=key)
    print(f"  Found {ad_sub.obs[key].nunique()} subclusters at resolution {res}")
    ad_sub.write_h5ad(outdir_h5ad + f"{ct}_subclustered.h5ad", compression="gzip")
    print(f"  Saved: {ct}_subclustered.h5ad")

ann_comb.write_h5ad(outdir_h5ad + 'xenium_integrated.h5ad', compression='gzip')
print(f"Saved integrated dataset: {ann_comb.n_obs} cells x {ann_comb.n_vars} features")